<a href="https://colab.research.google.com/github/developer-amna/Generative-AI/blob/main/Data_cleaning_%26_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import re
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
dataset = load_dataset("tweet_eval", "sentiment")

# Take only small subset (100 samples)
small_dataset = dataset["train"].shuffle(seed=42).select(range(100))

print(small_dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'text': 'Few more hours to iPhone 6s launch and im still using the 4th generation ^_^', 'label': 2}


In [ ]:
def clean_text(text):
    text = re.sub(r"http\S+", "", text)       # remove URLs
    text = re.sub(r"@\w+", "", text)          # remove mentions
    text = re.sub(r"#\w+", "", text)          # remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)   # remove special chars
    text = text.lower().strip()               # normalize
    return text

In [ ]:
for i in range(5):
    original = small_dataset[i]["text"]
    cleaned = clean_text(original)

    print("ORIGINAL :", original)
    print("CLEANED  :", cleaned)
    print("-"*50)

ORIGINAL : Few more hours to iPhone 6s launch and im still using the 4th generation ^_^
CLEANED  : few more hours to iphone s launch and im still using the th generation
--------------------------------------------------
ORIGINAL : Last night we were named NZ's 27th fastest growing co. in the Deloitte Fast 50. Our 2nd year making the list and we are totally thrilled!
CLEANED  : last night we were named nzs th fastest growing co in the deloitte fast  our nd year making the list and we are totally thrilled
--------------------------------------------------
ORIGINAL : All the hoes will be out this Saturday at the Chris brown concert.
CLEANED  : all the hoes will be out this saturday at the chris brown concert
--------------------------------------------------
ORIGINAL : BUENOS AIRES--Argentina late Wednesday approved a law to lower the legal voting age to 16 in a move that could s ...
CLEANED  : buenos airesargentina late wednesday approved a law to lower the legal voting age to  in a mov

In [ ]:
def create_prompt(example):
    text = clean_text(example["text"])
    label = example["label"]

    if label == 0:
        response = "I’m sorry to hear that. We will improve."
    elif label == 1:
        response = "Thank you for your message. We will look into it."
    else:
        response = "Thank you for your kind words!"

    return {
        "input_text": f"generate a polite response: {text}",
        "target_text": response
    }

formatted_dataset = small_dataset.map(create_prompt)

print(formatted_dataset[0])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

{'text': 'Few more hours to iPhone 6s launch and im still using the 4th generation ^_^', 'label': 2, 'input_text': 'generate a polite response: few more hours to iphone s launch and im still using the th generation', 'target_text': 'Thank you for your kind words!'}


In [ ]:
model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def tokenize(example):
    inputs = tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

    targets = tokenizer(
        example["target_text"],
        padding="max_length",
        truncation=True,
        max_length=32
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = formatted_dataset.map(tokenize, batched=False)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
split = tokenized_dataset.train_test_split(test_size=0.2)

train_dataset = split["train"]
eval_dataset = split["test"]

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,12.862135
20,8.469504
30,6.271863


TrainOutput(global_step=30, training_loss=9.20116729736328, metrics={'train_runtime': 95.7214, 'train_samples_per_second': 2.507, 'train_steps_per_second': 0.313, 'total_flos': 4060254044160.0, 'train_loss': 9.20116729736328, 'epoch': 3.0})

In [ ]:
def generate_response(text):
    cleaned = clean_text(text)
    input_text = f"generate a polite response: {cleaned}"

    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_length=40
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Load fresh model for BEFORE comparison
base_model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)

def generate_before(text):
    cleaned = clean_text(text)
    input_text = f"generate a polite response: {cleaned}"

    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    outputs = base_model.generate(
        **inputs,
        max_length=40
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


for i in range(5):
    sample = small_dataset[i]["text"]

    before = generate_before(sample)
    after = generate_response(sample)

    print("INPUT TEXT :", sample)
    print("BEFORE     :", before)
    print("AFTER      :", after)
    print("="*60)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

INPUT TEXT : Few more hours to iPhone 6s launch and im still using the 4th generation ^_^
BEFORE     : a polite response: few more hours to iphone s launch and im still using the th generation th generation
AFTER      : a polite response: a few hours to iphone s launch and im using the th generation s launch.
INPUT TEXT : Last night we were named NZ's 27th fastest growing co. in the Deloitte Fast 50. Our 2nd year making the list and we are totally thrilled!
BEFORE     : nzs th fastest growing co in the deloitte fast our nd year making the list and we are totally thrilled with the list.
AFTER      : nzs th fastest growing co in the deloitte fast.
INPUT TEXT : All the hoes will be out this Saturday at the Chris brown concert.
BEFORE     : a polite response: all the hoes will be out this saturday at the chris brown concert at the chris brown concert.
AFTER      : chris brown concert - chris brown - saturday - saturday -.
INPUT TEXT : BUENOS AIRES--Argentina late Wednesday approved a law t